# 🏋️ Train your from-scratch model on a FREE GPU — two-stage pipeline

The same recipe already proven on CPU in the repo, scaled up:

- **Stage A — Pretraining**: ~40 classic books + real Python source code
  (Flask, pytest, Requests, ...) teach English AND code structure.
- **Stage B — Chat fine-tune**: chat-heavy mix at low LR teaches assistant
  behavior, code-writing answers and honest "I don't know" — without
  erasing Stage A.

From **random weights** — no pretrained models, no AI APIs. Result runs on
a phone (Termux, pure NumPy) in <200 MB RAM.

**Before running:** menu **Runtime → Change runtime type → T4 GPU → Save.**
Then press ▶ on each step in order. Stage A is ~2–4 h; everything else is minutes.
Re-run Step 4 any time to train more — it resumes from the last checkpoint.

**Honest expectations:** GPT-2-nano-class output — fluent-ish English,
code-shaped Python, correct answers on trained patterns. Not ChatGPT;
visibly better than the 4M CPU model.

In [ ]:
#@title Step 1 — get the code, check the GPU { display-mode: "form" }
import os, sys, torch
REPO = "https://github.com/debzitsu-ship-it/Project-lmarena.git"
BRANCH = "arena/01a0011c-project-lmarena"
if not os.path.exists("Project-lmarena"):
    !git clone -q -b {BRANCH} {REPO}
%cd -q /content/Project-lmarena
!git pull -q
sys.path.insert(0, "/content/Project-lmarena")
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
print("✅ GPU:", torch.cuda.get_device_name(0))

In [ ]:
#@title Step 2 — download training data (books + Python code + chat) { display-mode: "form" }
# Books via GITenberg mirrors (public domain) + real Python repos (MIT/BSD).
# Colab has open internet, so extra Gutenberg books are fetched directly too.
import os, urllib.request
!python -m my_ai.data.fetch_gitenberg
!python -m my_ai.data.fetch_python_code

EXTRA_GUTENBERG_IDS = [1080, 2542, 5200, 16389, 244, 902, 768, 408, 1232, 844]
for bid in EXTRA_GUTENBERG_IDS:
    dest = f"my_ai/data/raw/gutenberg_{bid}.txt"
    if os.path.exists(dest): continue
    for url in (f"https://www.gutenberg.org/cache/epub/{bid}/pg{bid}.txt",
                f"https://www.gutenberg.org/files/{bid}/{bid}-0.txt"):
        try:
            txt = urllib.request.urlopen(url, timeout=30).read().decode("utf-8", "replace")
            s = txt.find("*** START"); e = txt.find("*** END")
            if s != -1: txt = txt[txt.find("\n", s):]
            if e != -1: txt = txt[:e]
            open(dest, "w").write(txt); break
        except Exception: continue

# MY_URLS: add any web pages YOU want the model to learn from
MY_URLS = []
if MY_URLS:
    from my_ai.data.ingest_urls import fetch_url
    import hashlib
    for u in MY_URLS:
        try:
            t = fetch_url(u)
            open(f"my_ai/data/raw/web_{hashlib.sha1(u.encode()).hexdigest()[:10]}.txt", "w").write(t)
            print("OK", u)
        except Exception as ex: print("SKIP", u, ex)

!python -m my_ai.data.make_chat_data
total = sum(os.path.getsize(f"my_ai/data/raw/{f}") for f in os.listdir("my_ai/data/raw"))
print(f"corpus size: {total/1e6:.1f} MB")

In [ ]:
#@title Step 3 — train tokenizer + pack tokens (~15–30 min) { display-mode: "form" }
!python -m my_ai.prepare_data --input my_ai/data/raw --out my_ai/data/processed \
    --vocab-size 8192 --tokenizer-sample-chars 2000000
!python -m my_ai.prepare_finetune --data my_ai/data/processed --raw my_ai/data/raw --chat-frac 0.7

In [ ]:
#@title Step 4 — STAGE A: pretrain on GPU (~2–4 h; re-run to train MORE) { display-mode: "form" }
import os
resume = "--resume my_ai/checkpoints/latest.pt" if os.path.exists("my_ai/checkpoints/latest.pt") else ""
!python -m my_ai.train --config my_ai/configs/small_20m.json \
    --data my_ai/data/processed --steps 20000 --batch-size 32 --lr 6e-4 {resume} \
    --sample-prompt "def add(a, b):"
print("\nStage A done. Re-run this cell to continue pretraining, or go to Step 5.")

In [ ]:
#@title Step 5 — STAGE B: chat fine-tune (~15 min) { display-mode: "form" }
!python -m my_ai.finetune --base my_ai/checkpoints/latest.pt \
    --data my_ai/data/processed --steps 2000 --lr 1e-4 --batch-size 32
print("\nChat model saved to my_ai/checkpoints/chat/best.pt")

In [ ]:
#@title Step 6 — chat with your newly trained model { display-mode: "form" }
import sys
sys.path.insert(0, "/content/Project-lmarena")
from my_ai.training.trainer import load_checkpoint, pick_device
from my_ai.tokenizer.tokenizer import load_tokenizer, EOS, USER_TOK, ASSISTANT_TOK
from my_ai.inference.generate import generate
from my_ai.chat.cli import build_prompt_ids

device = pick_device()
model, ckpt = load_checkpoint("my_ai/checkpoints/chat/best.pt", device=device)
model.eval()
tokenizer = load_tokenizer("my_ai/checkpoints/chat/tokenizer.json")
print(f"chatting with step-{ckpt.get('step')} chat model. 'quit' to stop.\n")
history = []
while True:
    msg = input("You: ").strip()
    if not msg: continue
    if msg.lower() in ("quit", "exit"): break
    ids = build_prompt_ids(tokenizer, [], history, msg, model.cfg.context_length)
    out = generate(model, ids, max_new_tokens=200, temperature=0.7, top_k=50,
                   top_p=0.95, repetition_penalty=1.15,
                   stop_tokens={EOS, USER_TOK, ASSISTANT_TOK}, device=device)
    reply = tokenizer.decode(out).strip()
    print("AI:", reply, "\n")
    history.append({"role": "user", "text": msg}); history.append({"role": "assistant", "text": reply})

In [ ]:
#@title Step 7 — export for your phone (Termux) and download { display-mode: "form" }
!python -m my_ai.inference.export_numpy --checkpoint my_ai/checkpoints/chat/best.pt \
    --out my_ai/checkpoints/model_numpy.npz
!cp my_ai/checkpoints/chat/tokenizer.json my_ai/checkpoints/tokenizer.json
from google.colab import files
files.download("my_ai/checkpoints/model_numpy.npz")
files.download("my_ai/checkpoints/tokenizer.json")
print("Put both files into Project-lmarena/my_ai/checkpoints/ on your phone,")
print("then run:  python -m my_ai.chat.termux_chat")

### Training *more and more*
- **Re-run Step 4** to keep pretraining (auto-resumes), then re-run Step 5 to re-polish chat.
- **More code expertise**: add repos to `REPOS` in `my_ai/data/fetch_python_code.py`,
  re-run Steps 2–5. More/better code data is the single biggest lever.
- **More books/URLs**: extend Step 2 lists, re-run Steps 2–5.
  ⚠️ Changing the corpus changes the tokenizer → delete old checkpoints first
  (`!rm -rf my_ai/checkpoints/*.pt my_ai/checkpoints/chat`) — a model is tied to its tokenizer.
- **Save between sessions**: Colab wipes files when the runtime dies — use Step 7
  or mount Google Drive and copy `my_ai/checkpoints/` there.
- **Bigger model**: swap `small_20m.json` → `base_50m.json` in Step 4 (needs longer runs).